## Download data from gdrive

In [14]:
! pip install -q -U gdown

import gdown

audio_url = "https://drive.google.com/file/d/1Rp43l85A8SGd7gNN3Xf07fz-pE5tJyZF/view?usp=sharing"
transcript_url = "https://drive.google.com/file/d/1BPZz40Lmul61m_aBBI8d13Wwm7ZfIUFm/view?usp=sharing"
output_audio = "stt_audio.zip"
output_transcript= "stt_transcript.zip"

downloaded_transcript = gdown.download(
    url=transcript_url,
    output=output_transcript,
    quiet=False,
)
downloaded_file_audio = gdown.download(
    url=audio_url,
    output=output_audio,
    quiet=False,
)
print("Downloaded to:", downloaded_file_audio, downloaded_transcript)

Downloading...
From: https://drive.google.com/uc?id=1BPZz40Lmul61m_aBBI8d13Wwm7ZfIUFm
To: /mnt/batch/tasks/shared/LS_root/mounts/clusters/computer-vision-team-i2/code/Users/kuladeep.a/STT_finetuning/stt_transcript.zip
100%|██████████| 1.70M/1.70M [00:00<00:00, 172MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1Rp43l85A8SGd7gNN3Xf07fz-pE5tJyZF
From (redirected): https://drive.google.com/uc?id=1Rp43l85A8SGd7gNN3Xf07fz-pE5tJyZF&confirm=t&uuid=8c5ae2e5-c688-4bc7-b509-e3d42fe5bd96
To: /mnt/batch/tasks/shared/LS_root/mounts/clusters/computer-vision-team-i2/code/Users/kuladeep.a/STT_finetuning/stt_audio.zip
100%|██████████| 595M/595M [00:12<00:00, 47.2MB/s] 


Downloaded to: stt_audio.zip stt_transcript.zip


In [ ]:
! unzip /home/azureuser/cloudfiles/code/Users/kuladeep.a/STT_finetuning/stt_audio.zip 
! unzip /home/azureuser/cloudfiles/code/Users/kuladeep.a/STT_finetuning/stt_transcript.zip
! mkdir -p stt_audio/audio stt_audio/transcribe_raw
! mv ./*.mp3 ./stt_audio/audio/
! mv ./transcripts_raw/*.json ./stt_audio/transcribe_raw/
! rm -rf ./transcripts_raw ./stt_transcript.zip ./stt_audio.zip

## Transcribe audio files using whisper-large-v3

In [ ]:
# ==============================================================================
#  LOCAL AUDIO -> WHISPER large-v3 TRANSCRIPTION
# ==============================================================================
#
#  IN : ./stt_audio/            (any folder of audio files; set CFG['audio_dir'])
#  OUT: ./stt_pilot/transcripts_raw/<id>.json   (segments + word timestamps)
#
#  No download, no unzip. Point it at a folder that already has audio in it.
#
#  Resumable: re-run any time; ids already in transcripts_raw are skipped.
# ==============================================================================

import json
import time
from pathlib import Path

try:
    from tabulate import tabulate
except ImportError:
    tabulate = None


# ------------------------------------------------------------------- CONFIG
CFG = {
    # --- input -------------------------------------------------------------
    "audio_dir":   "./stt_audio",     # folder containing the audio files
    "recursive":   True,              # scan subfolders too
    "audio_exts":  (".mp3", ".wav", ".m4a", ".mp4", ".webm", ".flac", ".ogg"),

    # --- output ------------------------------------------------------------
    "out_dirname": "stt_pilot",       # -> ./stt_pilot/transcripts_raw/

    # --- whisper -----------------------------------------------------------
    "model":          "large-v3",
    "language":       "en",
    "beam_size":      5,
    "min_silence_ms": 500,
    "limit":          None,     # 5 for a sanity run, None for everything
}

CWD       = Path.cwd()
AUDIO_DIR = Path(CFG["audio_dir"]).expanduser().resolve()
OUT_ROOT  = CWD / CFG["out_dirname"]
RAW_DIR   = OUT_ROOT / "transcripts_raw"
CLEAN_DIR = OUT_ROOT / "transcripts_clean"


CORE_TERMS = [
    "quarter panel", "fender", "bumper", "bumper cover", "bonnet", "hood",
    "unibody", "chassis", "crumple zone", "radiator", "headlight", "tail light",
    "windshield", "windscreen", "airbag", "frame damage", "structural damage",
    "rocker panel", "a-pillar", "b-pillar", "dent", "crease", "scratch", "scuff",
    "paintless dent repair", "salvage title", "write off", "total loss",
    "actual cash value", "diminished value", "deductible", "adjuster",
    "appraisal", "estimate", "teardown", "OEM", "aftermarket",
]
# Whisper truncates initial_prompt at 224 tokens and drops the tail silently,
# so keep the highest-value terms first.
INITIAL_PROMPT = ("Automotive collision damage assessment and insurance claims. "
                  "Terms: " + ", ".join(CORE_TERMS) + ".")


# ------------------------------------------------------------------ HELPERS
def _audio_files(root=None):
    root = root or AUDIO_DIR
    if not root.exists():
        return []
    it = root.rglob("*") if CFG["recursive"] else root.glob("*")
    return sorted(p for p in it
                  if p.is_file() and p.suffix.lower() in CFG["audio_exts"])


def _show(rows, headers):
    if tabulate:
        print(tabulate(rows, headers=headers, tablefmt="github"))
    else:
        for r in rows:
            print("  " + "  ".join(str(x) for x in r))


def _out_path(path):
    """Output json for an audio file. Uses the stem, so filenames must be
       unique across subfolders when recursive=True."""
    return RAW_DIR / (path.stem + ".json")


# ------------------------------------------------------------------ INVENTORY
def prepare():
    """Create output dirs and report what is there."""
    for d in (RAW_DIR, CLEAN_DIR):
        d.mkdir(parents=True, exist_ok=True)

    if not AUDIO_DIR.exists():
        raise SystemExit("audio dir not found: {}".format(AUDIO_DIR))

    files = _audio_files()
    if not files:
        raise SystemExit("no audio under {} (exts={}, recursive={})".format(
            AUDIO_DIR, CFG["audio_exts"], CFG["recursive"]))

    # duplicate stems would overwrite each other's output
    stems = [p.stem for p in files]
    dupes = {s for s in stems if stems.count(s) > 1}
    if dupes:
        print("WARNING: {} duplicate filename stems -> outputs will collide: {}"
              .format(len(dupes), sorted(dupes)[:5]))

    by_ext = {}
    for p in files:
        by_ext[p.suffix.lower()] = by_ext.get(p.suffix.lower(), 0) + 1
    mb = sum(p.stat().st_size for p in files) / 1e6
    already = sum(1 for p in files if _out_path(p).exists())

    _show([["audio files", len(files)],
           ["by extension", by_ext],
           ["total size", "{:.1f} MB".format(mb)],
           ["audio dir", str(AUDIO_DIR)],
           ["output dir", str(RAW_DIR)],
           ["already transcribed", already],
           ["remaining", len(files) - already]],
          headers=["", "value"])
    return files


# -------------------------------------------------------------------- WHISPER
def load_whisper(cpu=False):
    from faster_whisper import WhisperModel
    return WhisperModel(CFG["model"],
                        device="cpu" if cpu else "cuda",
                        compute_type="int8" if cpu else "float16")


def transcribe_one(model, path):
    segments, info = model.transcribe(
        str(path),
        language=CFG["language"],
        vad_filter=True,                          # kills hallucination on silence
        vad_parameters=dict(min_silence_duration_ms=CFG["min_silence_ms"]),
        initial_prompt=INITIAL_PROMPT,            # domain vocabulary bias
        word_timestamps=True,                     # needed for frame pairing
        beam_size=CFG["beam_size"],
    )
    segs = [{
        "start": round(s.start, 2), "end": round(s.end, 2),
        "text": s.text.strip(),
        "avg_logprob": round(s.avg_logprob, 3),
        "no_speech_prob": round(s.no_speech_prob, 3),
        "words": [{"w": w.word, "s": round(w.start, 2), "e": round(w.end, 2)}
                  for w in (s.words or [])],
    } for s in segments]
    return segs, info


def transcribe_all(cpu=False, limit=None):
    """Sequential (GPU-bound). Resumable: skips ids already in RAW_DIR."""
    files = _audio_files()
    pending = [p for p in files if not _out_path(p).exists()]
    if limit:
        pending = pending[:limit]
    if not pending:
        print("nothing to do -- all {} files already transcribed".format(len(files)))
        return []

    print("\ntranscribing {} of {} files".format(len(pending), len(files)))
    model = load_whisper(cpu)
    rows, done, failed = [], 0, []

    for path in pending:
        t0 = time.time()
        try:
            segs, info = transcribe_one(model, path)
        except Exception as e:
            print("  FAIL {}: {}".format(path.stem, e))
            failed.append(path.stem)
            continue

        mins = round(sum(s["end"] - s["start"] for s in segs) / 60, 1)
        _out_path(path).write_text(json.dumps({
            "id": path.stem, "source_file": str(path),
            "language": info.language, "speech_minutes": mins,
            "n_segments": len(segs), "segments": segs,
        }, indent=2))

        done += 1
        el = time.time() - t0
        rows.append([path.stem[:34], mins, len(segs), "{:.0f}s".format(el)])
        print("  [{}/{}] {:34s} {:>5} min  {:>4} segs  {:.0f}s".format(
            done, len(pending), path.stem[:34], mins, len(segs), el))

    print()
    _show(rows, headers=["id", "min", "segs", "elapsed"])
    total = sum(r[1] for r in rows)
    print("\ntranscribed {} files, {:.1f} min ({:.2f} h) of speech".format(
        done, total, total / 60))
    if failed:
        print("failed ({}): {}".format(len(failed), ", ".join(failed[:10])))
    print("output: {}".format(RAW_DIR))
    return rows


# ------------------------------------------------------------------------ RUN
if __name__ == "__main__":
    prepare()
    transcribe_all(limit=CFG["limit"])

## Data Validation

In [20]:
# ==============================================================================
#  DATA VALIDATION  --  aligned transcript player (listen + read, synced)
# ==============================================================================
#  Pairs each audio file with its transcript and indexes them, so you review by
#  position rather than by pasting ids:
#
#      index()      -> print the paired list with an idx column
#      play(0)      -> load and play item 0
#      play("yt_abc123")   -> still works if you prefer the id
#      play(3, only_low=True)  -> show ONLY low-confidence segments
#
#  Click any line to seek. The playing segment highlights and auto-scrolls.
#  Domain terms are marked so you can hear whether Whisper actually got them.
#
#  Audio is re-encoded to low-bitrate mono mp3 before base64-embedding, so a
#  30-minute file stays a few MB instead of blowing up the notebook.
# ==============================================================================

import base64
import html
import json
import re
import subprocess
import tempfile
from pathlib import Path

from IPython.display import HTML, display

try:
    from tabulate import tabulate
except ImportError:
    tabulate = None


# ------------------------------------------------------------------- CONFIG

BASE      = Path("/home/azureuser/cloudfiles/code/Users/kuladeep.a/STT_finetuning/stt_audio")
AUDIO_SRC = BASE / "audio"
RAW_DIR   = BASE / "transcribe_raw"
CLEAN_DIR = BASE / "transcribe_clean"      # may not exist yet; that's fine

CACHE_DIR = Path("/home/azureuser/cloudfiles/code/Users/kuladeep.a/STT_finetuning")   # local disk, NOT cloudfiles

PLAYER = {
    "bitrate":  "32k",     # preview quality only; keeps base64 small
    "lp_warn":  -0.6,      # flag segments below this avg_logprob
    "max_min":  None,      # e.g. 10 -> only embed the first 10 minutes
    "recursive": True,     # search AUDIO_SRC subfolders
}

CORE_TERMS = [
    "quarter panel", "fender", "bumper", "bumper cover", "bonnet", "hood",
    "unibody", "chassis", "crumple zone", "radiator", "headlight", "tail light",
    "windshield", "windscreen", "airbag", "frame damage", "structural damage",
    "rocker panel", "a-pillar", "b-pillar", "dent", "crease", "scratch", "scuff",
    "paintless dent repair", "salvage title", "write off", "total loss",
    "actual cash value", "diminished value", "deductible", "adjuster",
    "appraisal", "estimate", "teardown", "OEM", "aftermarket", "axle",
    "rear-ended",
]

AUDIO_EXTS = (".mp3", ".wav", ".m4a", ".mp4", ".webm", ".flac", ".ogg")

ITEMS = []          # populated by index(); list of dicts


# ------------------------------------------------------------------- INDEX
def _find_audio(doc_id):
    it = AUDIO_SRC.rglob("*") if PLAYER["recursive"] else AUDIO_SRC.glob("*")
    for p in sorted(it):
        if p.is_file() and p.stem == doc_id and p.suffix.lower() in AUDIO_EXTS:
            return p
    return None


def _transcript_path(doc_id):
    """Prefer the cleaned doc (has verdict + text_original), fall back to raw."""
    for d in (CLEAN_DIR, RAW_DIR):
        p = d / (doc_id + ".json")
        if p.exists():
            return p, d.name
    return None, None


def index(show=True, missing=False):
    """Build the paired [audio, transcript] list. Returns ITEMS."""
    global ITEMS
    ITEMS = []
    orphans = []

    if not RAW_DIR.exists():
        raise SystemExit("transcript dir not found: {}".format(RAW_DIR))

    doc_ids = sorted({p.stem for p in RAW_DIR.glob("*.json")} |
                     {p.stem for p in CLEAN_DIR.glob("*.json")}
                     if CLEAN_DIR.exists() else
                     {p.stem for p in RAW_DIR.glob("*.json")})

    for did in doc_ids:
        tpath, tsrc = _transcript_path(did)
        apath = _find_audio(did)
        if apath is None:
            orphans.append(did)
            continue

        doc = json.loads(tpath.read_text())
        segs = doc.get("segments", [])
        lows = sum(1 for s in segs
                   if s.get("avg_logprob", 0.0) < PLAYER["lp_warn"])
        ITEMS.append({
            "idx": len(ITEMS),
            "id": did,
            "audio": apath,
            "transcript": tpath,
            "src": tsrc,
            "minutes": doc.get("speech_minutes", 0.0),
            "n_segments": len(segs),
            "n_low": lows,
            "pct_low": round(100.0 * lows / len(segs), 1) if segs else 0.0,
        })

    if show:
        rows = [[i["idx"], i["id"][:30], i["minutes"], i["n_segments"],
                 i["n_low"], "{}%".format(i["pct_low"]), i["src"]]
                for i in ITEMS]
        hdr = ["idx", "id", "min", "segs", "low", "%low", "from"]
        if tabulate:
            print(tabulate(rows, headers=hdr, tablefmt="github"))
        else:
            print("  ".join(hdr))
            for r in rows:
                print("  ".join(str(x) for x in r))
        print("\n{} paired items".format(len(ITEMS)))
        if orphans:
            print("{} transcripts with NO matching audio{}".format(
                len(orphans), ": " + ", ".join(orphans[:5]) if missing else
                " (call index(missing=True) to list)"))
    return ITEMS


def _resolve(ref):
    """Accept an int index or an id string."""
    if not ITEMS:
        index(show=False)
    if isinstance(ref, int):
        if not 0 <= ref < len(ITEMS):
            raise IndexError("idx {} out of range (0..{})".format(ref, len(ITEMS) - 1))
        return ITEMS[ref]
    for it in ITEMS:
        if it["id"] == ref:
            return it
    raise KeyError("no indexed item with id {!r} -- run index() first".format(ref))


# ------------------------------------------------------------------- PREVIEW
def _preview_mp3(item):
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    out = CACHE_DIR / "{}.mp3".format(item["id"])
    if out.exists():
        return out
    cmd = ["ffmpeg", "-nostdin", "-y", "-loglevel", "error", "-i", str(item["audio"])]
    if PLAYER["max_min"]:
        cmd += ["-t", str(PLAYER["max_min"] * 60)]
    cmd += ["-ac", "1", "-b:a", PLAYER["bitrate"], str(out)]
    subprocess.run(cmd, check=True)
    return out


def _mark_terms(text):
    esc = html.escape(text)
    for t in sorted(CORE_TERMS, key=len, reverse=True):
        esc = re.sub(r"(?i)\b({})\b".format(re.escape(t)),
                     r'<span class="tm">\1</span>', esc)
    return esc


# ---------------------------------------------------------------------- PLAY
def play(ref=0, show_original=True, only_low=False):
    """ref = integer index from index(), or a document id string."""
    item = _resolve(ref)
    doc = json.loads(item["transcript"].read_text())
    mp3 = _preview_mp3(item)
    b64 = base64.b64encode(mp3.read_bytes()).decode()

    dom = doc.get("domain", {})
    meta = ("[{}] {} &middot; {} min &middot; {} segments &middot; "
            "{} low ({}%) &middot; {}").format(
        item["idx"], doc["id"], item["minutes"], item["n_segments"],
        item["n_low"], item["pct_low"], item["src"])
    if dom:
        meta += " &middot; tier <b>{}</b> &middot; {}".format(
            dom.get("tier"), dom.get("audio_setting"))
    if only_low:
        meta += ' &middot; <b>showing low-confidence only</b>'

    rows = []
    for i, s in enumerate(doc["segments"]):
        lp = s.get("avg_logprob", 0.0)
        is_low = lp < PLAYER["lp_warn"]
        if only_low and not is_low:
            continue
        cls = "seg" + (" low" if is_low else "")
        orig = s.get("text_original")
        changed = show_original and orig and orig != s["text"]
        body = _mark_terms(s["text"])
        if changed:
            body += '<div class="orig">was: {}</div>'.format(html.escape(orig))
        rows.append(
            '<div class="{}" data-s="{:.2f}" data-e="{:.2f}" id="sg{}">'
            '<span class="ts">{:>7.1f}</span>'
            '<span class="lp">{:+.2f}</span>'
            '<span class="tx">{}</span></div>'.format(
                cls, s["start"], s["end"], i, s["start"], lp, body))

    if not rows:
        rows = ['<div class="seg"><span class="tx">'
                '(no segments to show)</span></div>']

    tpl = """
<style>
  .wrap {{ font-family: ui-monospace, Menlo, monospace; font-size: 13px; }}
  .meta {{ padding: 6px 0; color: #555; }}
  .list {{ max-height: 460px; overflow-y: auto; border: 1px solid #ddd;
           border-radius: 6px; padding: 4px; }}
  .seg  {{ display: flex; gap: 10px; padding: 5px 7px; border-radius: 4px;
           cursor: pointer; align-items: baseline; }}
  .seg:hover {{ background: #f2f6ff; }}
  .seg.on    {{ background: #fff3bf; }}
  .seg.low   {{ border-left: 3px solid #e8a33d; }}
  .ts   {{ color: #888; flex: 0 0 58px; }}
  .lp   {{ color: #aaa; flex: 0 0 44px; }}
  .tx   {{ flex: 1; line-height: 1.45; }}
  .tm   {{ background: #d3f9d8; border-radius: 3px; padding: 0 2px; }}
  .orig {{ color: #c92a2a; font-size: 11.5px; margin-top: 2px; }}
  audio {{ width: 100%; margin: 6px 0; }}
</style>
<div class="wrap">
  <div class="meta">{meta}</div>
  <audio id="au" controls src="data:audio/mpeg;base64,{b64}"></audio>
  <div class="list" id="ls">{rows}</div>
</div>
<script>
(function() {{
  var au = document.getElementById('au');
  var ls = document.getElementById('ls');
  var segs = Array.prototype.slice.call(ls.querySelectorAll('.seg'));
  segs.forEach(function(el) {{
    el.addEventListener('click', function() {{
      if (!el.dataset.s) return;
      au.currentTime = parseFloat(el.dataset.s);
      au.play();
    }});
  }});
  var last = null;
  au.addEventListener('timeupdate', function() {{
    var t = au.currentTime, hit = null;
    for (var i = 0; i < segs.length; i++) {{
      if (!segs[i].dataset.s) continue;
      if (t >= parseFloat(segs[i].dataset.s) && t < parseFloat(segs[i].dataset.e)) {{
        hit = segs[i]; break;
      }}
    }}
    if (hit !== last) {{
      if (last) last.classList.remove('on');
      if (hit) {{
        hit.classList.add('on');
        var top = hit.offsetTop - ls.offsetTop;
        if (top < ls.scrollTop || top > ls.scrollTop + ls.clientHeight - 60) {{
          ls.scrollTop = top - ls.clientHeight / 3;
        }}
      }}
      last = hit;
    }}
  }});
}})();
</script>
"""
    display(HTML(tpl.format(meta=meta, b64=b64, rows="".join(rows))))
    return item


def next_item(step=1):
    """Advance through the list: play(0), then next_item(), next_item(), ..."""
    if not ITEMS:
        index(show=False)
    cur = getattr(next_item, "_cur", -1) + step
    next_item._cur = max(0, min(cur, len(ITEMS) - 1))
    return play(next_item._cur)


# ------------------------------------------------------------------------ RUN
# index()
play(10)

{'idx': 10,
 'id': 'yt_88L3KLkH9oI',
 'audio': PosixPath('/home/azureuser/cloudfiles/code/Users/kuladeep.a/STT_finetuning/stt_audio/audio/yt_88L3KLkH9oI.mp3'),
 'transcript': PosixPath('/home/azureuser/cloudfiles/code/Users/kuladeep.a/STT_finetuning/stt_audio/transcribe_raw/yt_88L3KLkH9oI.json'),
 'src': 'transcribe_raw',
 'minutes': 3.1,
 'n_segments': 22,
 'n_low': 0,
 'pct_low': 0.0}

## LLM- Data Cleaning

In [2]:
# ==============================================================================
#  LLM DOMAIN FILTER  --  classify only, never edit
# ==============================================================================
#  Reads each transcript, asks the LLM whether it is about vehicle damage /
#  auto insurance, and writes a verdict. Transcripts are NEVER modified:
#  no grammar fixes, no punctuation, no ASR-error correction. What Whisper
#  heard stays exactly as it is, because it is the label for the audio.
#
#  IN : <BASE>/transcribe_raw/<id>.json
#  OUT: <BASE>/verdicts.csv          one row per file, with paths
#       <BASE>/reject_paths.txt      audio+json paths you can review and delete
#       <BASE>/verdicts/<id>.json    per-file cache (makes re-runs resumable)
#
#  API KEY:  export CLAUDE_API="sk-ant-..."   before launching jupyter/python
# ==============================================================================

import csv
import json
import os
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

try:
    from tabulate import tabulate
except ImportError:
    tabulate = None


# ------------------------------------------------------------------- CONFIG
BASE      = Path("/home/azureuser/cloudfiles/code/Users/kuladeep.a/"
                 "STT_finetuning/stt_audio")
AUDIO_DIR = BASE / "audio"
RAW_DIR   = BASE / "transcribe_raw"
VERD_DIR  = BASE / "verdicts"
CSV_OUT   = BASE / "verdicts.csv"
REJECTS   = BASE / "reject_paths.txt"

CFG = {
    # haiku is plenty for a yes/no domain call and far cheaper across hundreds
    # of files; switch to claude-sonnet-5 if the judgements look sloppy
    "model":         "claude-haiku-4-5-20251001",
    "max_workers":   4,
    "excerpt_chars": 3000,
    "limit":         None,      # 5 for a sanity run
}

AUDIO_EXTS = (".mp3", ".wav", ".m4a", ".mp4", ".webm", ".flac", ".ogg")


# ------------------------------------------------------------------- CLIENT
def _client():
    from anthropic import Anthropic
    try:
        import config
    except ImportError:
        raise RuntimeError(
            "config.py not found. Create it next to this script:\n"
            "    CLAUDE_API = 'sk-ant-...'\n"
            "and add config.py to .gitignore.")
    key = getattr(config, "CLAUDE_API", "").strip()
    if not key:
        raise RuntimeError("config.CLAUDE_API is empty.")
    return Anthropic(api_key=key)


PROMPT = """You are screening transcripts for a speech dataset about VEHICLE DAMAGE
and AUTO INSURANCE. Some files are off-topic: random videos, other subjects,
or automated narration that got scraped by mistake.

Decide whether this transcript belongs in the dataset.

Return ONLY a JSON object. No preamble, no markdown fences:
{{
  "in_domain": true|false,
  "subject": "<what the audio is actually about, 2-5 words>",
  "synthetic_narration": true|false,
  "confidence": "high"|"medium"|"low",
  "reason": "<one short sentence>"
}}

IN DOMAIN: cars, trucks, SUVs, vans - their damage, parts, collision repair,
body work, salvage, write-offs, total loss, claims, appraisal, inspection.

NOT IN DOMAIN: motorcycles, RVs, boats, aircraft, roofing, agriculture,
consumer electronics, gaming or simulation footage, general talk with no
vehicle-damage content, or "write off" used to mean a TAX deduction.

Set synthetic_narration=true if the speech reads like text-to-speech or a
read-aloud article: uniform pacing, no disfluencies, no false starts.
Such files are usable for vocabulary but not for acoustic training.

Set confidence="low" if the excerpt is too short or too vague to judge.

TRANSCRIPT ID: {tid}
EXCERPT:
{excerpt}
"""


# ------------------------------------------------------------------ HELPERS
def _excerpt(segs, n_chars=None):
    """Head + middle sample - enough for the LLM to judge subject matter."""
    n_chars = n_chars or CFG["excerpt_chars"]
    txt = " ".join(s.get("text", "") for s in segs).strip()
    if len(txt) <= n_chars:
        return txt
    h, mid = n_chars // 2, len(txt) // 2
    return txt[:h] + "\n[...]\n" + txt[mid:mid + h]


def _json_from(resp):
    t = resp.content[0].text.strip()
    t = re.sub(r"^```(?:json)?|```$", "", t, flags=re.M).strip()
    return json.loads(t)


def _find_audio(tid):
    for p in sorted(AUDIO_DIR.rglob("*")):
        if p.is_file() and p.stem == tid and p.suffix.lower() in AUDIO_EXTS:
            return p
    return None


def _show(rows, headers):
    if tabulate:
        print(tabulate(rows, headers=headers, tablefmt="github"))
    else:
        print("  ".join(headers))
        for r in rows:
            print("  ".join(str(x) for x in r))


# --------------------------------------------------------------- CLASSIFY
def classify_one(client, f):
    doc = json.loads(f.read_text())
    segs = doc.get("segments", [])
    tid = doc.get("id", f.stem)

    excerpt = _excerpt(segs)
    if not excerpt:
        v = {"in_domain": False, "subject": "empty transcript",
             "synthetic_narration": False, "confidence": "high",
             "reason": "no text in transcript"}
    else:
        r = client.messages.create(
            model=CFG["model"], max_tokens=500,
            messages=[{"role": "user",
                       "content": PROMPT.format(tid=tid, excerpt=excerpt)}])
        v = _json_from(r)

    audio = _find_audio(tid)
    rec = {
        "id": tid,
        "in_domain": bool(v.get("in_domain")),
        "subject": v.get("subject", ""),
        "synthetic_narration": bool(v.get("synthetic_narration")),
        "confidence": v.get("confidence", ""),
        "reason": v.get("reason", ""),
        "speech_minutes": doc.get("speech_minutes", 0),
        "n_segments": len(segs),
        "audio_path": str(audio) if audio else "",
        "json_path": str(f),
    }
    (VERD_DIR / (tid + ".json")).write_text(json.dumps(rec, indent=2))
    return rec


def run_all(limit=None, max_workers=None):
    """Resumable: skips ids that already have a verdict cached."""
    max_workers = max_workers or CFG["max_workers"]
    VERD_DIR.mkdir(parents=True, exist_ok=True)

    if not RAW_DIR.exists():
        raise FileNotFoundError("transcript dir not found: {}".format(RAW_DIR))

    files = sorted(RAW_DIR.glob("*.json"))
    pending = [f for f in files if not (VERD_DIR / f.name).exists()]
    if limit:
        pending = pending[:limit]

    if pending:
        print("classifying {} of {} transcripts ({} workers, {})".format(
            len(pending), len(files), max_workers, CFG["model"]))
        client = _client()
        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            futs = {ex.submit(classify_one, client, f): f for f in pending}
            for i, fut in enumerate(as_completed(futs), 1):
                f = futs[fut]
                try:
                    rec = fut.result()
                except Exception as e:
                    print("  FAIL {}: {}".format(f.stem, e))
                    continue
                print("  [{}/{}] {:30s} {:4s} {:6s} {}".format(
                    i, len(pending), rec["id"][:30],
                    "KEEP" if rec["in_domain"] else "DROP",
                    rec["confidence"], rec["subject"][:40]))
    else:
        print("all {} transcripts already classified".format(len(files)))

    # collect every cached verdict, not just this run's
    rows = [json.loads(p.read_text()) for p in sorted(VERD_DIR.glob("*.json"))]
    _write_csv(rows)
    _write_rejects(rows)
    summary(rows)
    return rows


# ---------------------------------------------------------------- OUTPUTS
COLS = ["id", "in_domain", "subject", "synthetic_narration", "confidence",
        "speech_minutes", "n_segments", "reason", "audio_path", "json_path"]


def _write_csv(rows):
    with open(CSV_OUT, "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=COLS)
        w.writeheader()
        for r in sorted(rows, key=lambda x: (x["in_domain"], x["id"])):
            w.writerow({k: r.get(k, "") for k in COLS})
    print("\nwrote: {} ({} rows)".format(CSV_OUT, len(rows)))


def _write_rejects(rows):
    """Paths for the files the LLM flagged, so you can review and delete."""
    bad = [r for r in rows if not r["in_domain"]]
    low = [r for r in rows if r["in_domain"] and r["confidence"] == "low"]
    lines = ["# REJECTED - not in domain ({})".format(len(bad)),
             "# review these, then delete the ones you agree with", ""]
    for r in bad:
        lines.append("# {}  |  {}".format(r["subject"], r["reason"]))
        if r["audio_path"]:
            lines.append(r["audio_path"])
        lines.append(r["json_path"])
        lines.append("")
    lines += ["", "# LOW CONFIDENCE but kept ({}) - worth a listen".format(len(low)), ""]
    for r in low:
        lines.append("# {}  |  {}".format(r["subject"], r["reason"]))
        lines.append(r["json_path"])
        lines.append("")
    REJECTS.write_text("\n".join(lines))
    print("wrote: {} ({} rejected, {} low-confidence)".format(
        REJECTS, len(bad), len(low)))


def summary(rows):
    keep = [r for r in rows if r["in_domain"]]
    drop = [r for r in rows if not r["in_domain"]]
    tts = [r for r in keep if r["synthetic_narration"]]
    mk = sum(r.get("speech_minutes", 0) for r in keep)

    print()
    _show([["files", len(rows)],
           ["keep", len(keep)],
           ["drop", len(drop)],
           ["synthetic narration (kept)", len(tts)],
           ["in-domain audio", "{:.1f} min ({:.2f} h)".format(mk, mk / 60)]],
          ["", "value"])

    if drop:
        subj = {}
        for r in drop:
            subj[r["subject"]] = subj.get(r["subject"], 0) + 1
        print("\nwhy dropped:")
        _show(sorted(subj.items(), key=lambda x: -x[1])[:15],
              ["subject", "files"])

    if tts:
        print("\nNOTE: {} kept files look like synthetic narration. Usable for "
              "vocabulary, not for acoustic fine-tuning.".format(len(tts)))


def delete_rejected(dry_run=True):
    """Delete audio + json for every file marked not-in-domain.
       dry_run=True prints what WOULD go. Run the review first."""
    rows = [json.loads(p.read_text()) for p in sorted(VERD_DIR.glob("*.json"))]
    bad = [r for r in rows if not r["in_domain"]]
    n = 0
    for r in bad:
        for key in ("audio_path", "json_path"):
            p = r.get(key)
            if p and Path(p).exists():
                print(("would delete: " if dry_run else "deleted: ") + p)
                if not dry_run:
                    Path(p).unlink()
                n += 1
    print("\n{} {} files".format("would delete" if dry_run else "deleted", n))


# ------------------------------------------------------------------------ RUN
if __name__ == "__main__":
    run_all(limit=CFG["limit"])

classifying 86 of 86 transcripts (4 workers, claude-haiku-4-5-20251001)
  [1/86] yt_20wz75fqXKM                 KEEP high   Automotive collision repair certificatio
  [2/86] yt_-vPziwBXI0Q                 KEEP high   Vehicle damage assessment and repair
  [3/86] yt_1AV8aFoOTDc                 KEEP high   Paintless dent repair on BMW M3
  [4/86] yt_2BwR_RIoHDw                 KEEP high   Unibody stiffener installation and weldi
  [5/86] yt_2TKAhFVEgbU                 KEEP high   Car crash physics and vehicle crashworth
  [6/86] yt_54fe3AdbZyQ                 DROP high   empty transcript
  [7/86] yt_4IoTT0fHBOw                 KEEP high   Paintless dent removal repair
  [8/86] yt_3UVNxQPrhpk                 KEEP high   Salvaged vehicle rebuilding and inspecti
  [9/86] yt_88L3KLkH9oI                 KEEP high   Paintless dent repair demonstration
  [10/86] yt_7i9wBxy_VgE                 KEEP high   Vehicle collision damage assessment
  [11/86] yt_4KE4ujcnaTk                 KEEP high   Sa

In [7]:
import json, csv, datetime
from pathlib import Path

BASE  = Path("/home/azureuser/cloudfiles/code/Users/kuladeep.a/STT_finetuning/stt_audio")
AUDIO, RAW, VERD = BASE/"audio", BASE/"transcribe_raw", BASE/"verdicts"
LOG   = BASE / "removed_files.csv"

REMOVE_SUBJECTS = [
    "empty transcript",
    "Musical lyrics or singing",
    "Desert vehicle adventure vlog",
    "Shop management software tutorial",
    "Personal injury law advertisement",
]

DRY_RUN = False          # flip to False to actually delete

def _find(d, stem):
    return [p for p in d.iterdir() if p.is_file() and p.stem == stem] if d.exists() else []

rows = []
for vp in sorted(VERD.glob("*.json")):
    v = json.loads(vp.read_text())
    subj = (v.get("subject") or "").lower()
    hit = next((s for s in REMOVE_SUBJECTS if s.lower() in subj), None)
    if not hit:
        continue
    for p in _find(AUDIO, v["id"]) + _find(RAW, v["id"]) + [vp]:
        rows.append({"id": v["id"], "subject": v.get("subject"),
                     "matched": hit, "path": str(p),
                     "size_mb": round(p.stat().st_size/1e6, 2),
                     "removed_at": datetime.datetime.now().isoformat(timespec="seconds")})
        if not DRY_RUN:
            p.unlink()

if rows:
    new = not LOG.exists()
    with open(LOG, "a", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
        if new: w.writeheader()
        w.writerows(rows)

ids = sorted({r["id"] for r in rows})
print("{} {} files across {} ids".format(
    "WOULD DELETE" if DRY_RUN else "DELETED", len(rows), len(ids)))
for i in ids:
    print("  ", i, "-", next(r["subject"] for r in rows if r["id"] == i))
print("\nlog:", LOG)
print("remaining:", len(list(AUDIO.glob("*"))), "audio,",
      len(list(RAW.glob("*.json"))), "transcripts")

DELETED 27 files across 9 ids
   yt_54fe3AdbZyQ - empty transcript
   yt_CHkbp1a9Vms - Musical lyrics or singing
   yt_CKsTBBWohJU - empty transcript
   yt_OinkXuiPQFA - Desert vehicle adventure vlog
   yt_Rfgoetbr96E - Shop management software tutorial
   yt_akB1WqcEI48 - empty transcript
   yt_jBOitTAOqww - empty transcript
   yt_k-4y2oM8nvU - empty transcript
   yt_os9MC6HVliM - Personal injury law advertisement

log: /home/azureuser/cloudfiles/code/Users/kuladeep.a/STT_finetuning/stt_audio/removed_files.csv
remaining: 77 audio, 77 transcripts
